# Day 2 - Topic 6: Error Handling, Context Managers, Iterators vs Generators

> Lead-Level Data Science Interview Prep Series

## 1. Introduction

- **Error handling** (try/except) lets your program respond to failures gracefully instead of crashing
- **Context managers** (the with statement) guarantee setup/cleanup happens - files get closed, connections released - even if errors occur
- **Iterators** are objects you can loop over one item at a time; **generators** are the easiest way to create them (using yield)
- Why needed?
  - Real pipelines face bad files, missing keys, network failures - unhandled, one bad row kills a 6-hour job
  - Unclosed files/connections leak resources in long-running services
  - Generators let you stream datasets far larger than RAM
- Where used?
  - `with open(...)` - the most typed context manager in existence
  - try/except around every external interaction (files, APIs, databases) in production DS code
  - Reading a 50GB log file line by line via generators

## 2. Real-Life Analogy

- **try/except** = driving with airbags: you drive normally (try); IF a crash happens, the airbag deploys (except) instead of everything being destroyed; `finally` = the seatbelt-light check that runs no matter what happened
- **raise** = the smoke detector: it does not fix the fire, it loudly signals "something is wrong here" so the right responder can act
- **Context manager** = a hotel room keycard: entering (with) switches the lights on automatically; leaving ALWAYS switches them off - even if you left in a hurry (an exception). You cannot forget the cleanup
- **Iterator** = a Netflix series feed: you only get the "next episode" one at a time; **generator** = the show being filmed on demand - the next episode is produced only when you ask, and after the finale (StopIteration) there is nothing more

## 3. Explanation

- **Error handling flow:**
  - try: code that might fail
  - except SomeError: runs only if that error type occurred (catch specific types, not bare except)
  - else: runs only if NO error occurred (optional)
  - finally: ALWAYS runs - error or not (cleanup)
  - raise: throw an error yourself; custom exceptions = small classes inheriting from Exception
- **Context managers:**
  - `with open(f) as file:` calls `file.__enter__()` on entry and GUARANTEES `file.__exit__()` on exit (even on exceptions)
  - Any class implementing `__enter__` and `__exit__` works with with
- **Iterators vs Generators:**
  - Iterator protocol: `__iter__()` returns the iterator, `__next__()` returns the next value, raises StopIteration when done
  - A generator FUNCTION uses yield instead of return - calling it returns a generator object that implements the whole protocol automatically
  - yield PAUSES the function, remembering all local state; next() resumes from that exact point

> **Trick to remember:** try = attempt, except = rescue, finally = always. with = enter/exit guaranteed. yield = pause and hand over one value.

## 4. Syntax

```python
# Error handling - full form
try:
    risky_code()
except ValueError as e:          # catch a SPECIFIC error, bind it to e
    handle_it(e)
except (KeyError, TypeError):    # multiple types in one except
    handle_others()
else:
    runs_only_if_no_error()
finally:
    always_runs()

# Raising + custom exceptions
raise ValueError("Bad input")

class DataValidationError(Exception):    # custom exception = tiny class
    pass

# Context manager usage
with open("data.csv") as f:
    content = f.read()                   # f auto-closes after this block

# Custom context manager (class-based)
class Managed:
    def __enter__(self):
        # setup
        return self                      # bound to the as variable
    def __exit__(self, exc_type, exc_value, traceback):
        # cleanup - runs even on error
        return False                     # False = do not suppress exceptions

# Generator function
def count_up_to(n):
    i = 1
    while i <= n:
        yield i                          # pause here, emit i
        i += 1
```

In [ ]:
# All three in 12 lines
def safe_divide(a, b):
    try:
        return a / b
    except ZeroDivisionError:
        return None
    finally:
        pass  # cleanup would go here

print(safe_divide(10, 2), safe_divide(10, 0))

def firsts(n):
    for i in range(1, n + 1):
        yield i * i

gen = firsts(3)
print(next(gen), next(gen), next(gen))   # 1 4 9


## 5. Examples

### Basic Example

In [ ]:
# Basic: catching specific errors with meaningful responses
raw_values = ["42", "abc", "17"]

for v in raw_values:
    try:
        number = int(v)
        print(f"Parsed: {number}")
    except ValueError:
        print(f"Skipped bad value: {v!r}")


### Intermediate Example

In [ ]:
# Intermediate: custom exception + full try/except/else/finally + custom context manager
import time

class DataValidationError(Exception):
    """Raised when a record fails validation."""
    pass

def validate_age(age):
    if not isinstance(age, int):
        raise DataValidationError(f"Age must be int, got {type(age).__name__}")
    if not 0 <= age <= 120:
        raise DataValidationError(f"Age {age} outside human range")
    return age

class Timer:
    """Context manager that times its block - reusable forever."""
    def __enter__(self):
        self.start = time.time()
        return self
    def __exit__(self, exc_type, exc_value, traceback):
        self.elapsed = time.time() - self.start
        print(f"Block took {self.elapsed:.4f}s (error occurred: {exc_type is not None})")
        return False    # never swallow exceptions silently

with Timer():
    for age in [25, 150, 40]:
        try:
            validate_age(age)
        except DataValidationError as e:
            print("Rejected:", e)
        else:
            print("Accepted:", age)


- Custom exceptions carry MEANING - catching DataValidationError is precise; catching bare Exception is a shotgun
- else runs only for clean records - keeps the happy path visually separate from rescue logic
- The Timer context manager received exc_type in __exit__ - context managers KNOW whether their block failed (that is how they clean up correctly either way)
- Returning False from __exit__ lets real errors keep propagating - returning True would silently swallow them (rarely what you want)

### Real-World Example

In [ ]:
# Real-world: streaming a huge CSV with a generator - constant memory, robust to bad rows
def read_records(csv_lines):
    """Generator: yields clean dict records one at a time - never loads all rows."""
    headers = None
    for line_num, line in enumerate(csv_lines, start=1):
        try:
            parts = line.strip().split(",")
            if headers is None:
                headers = parts
                continue
            record = dict(zip(headers, parts))
            record["salary"] = float(record["salary"])     # may raise ValueError
            yield record                                    # emit ONE record, pause
        except ValueError:
            print(f"[warn] Skipping malformed line {line_num}: {line.strip()!r}")

# Simulating a file (in reality: with open("big.csv") as f: for rec in read_records(f))
fake_file = [
    "name,dept,salary",
    "Asha,Sales,52000",
    "Vikram,Engineering,not_a_number",
    "Neha,Sales,49000",
]

total = 0
for rec in read_records(fake_file):
    total += rec["salary"]

print("Total clean salary:", total)


- One bad row prints a warning and the pipeline KEEPS GOING - try/except inside the loop is what separates production code from notebook code
- The generator yields one record at a time - a 50GB file would use the same few KB of memory as this 4-line one
- File objects themselves are iterators (for line in f streams lazily) - so generator + open() composes into a full streaming pipeline
- This single function demonstrates all three topics working together: errors handled, resources managed, data streamed

## 6. Internal Working

- When an exception is raised, Python unwinds the call stack frame by frame looking for a matching except block; if none is found, the program exits with a traceback
- The with statement compiles to try/finally underneath - __exit__ in the finally position is WHY cleanup is guaranteed
- __exit__ receives (exc_type, exc_value, traceback) - all None on clean exit; returning True suppresses the exception
- Calling a generator function runs NOTHING - it instantly returns a generator object; each next() executes up to the next yield, then freezes the entire frame (locals, position) until the following next()
- When the function ends, the generator raises StopIteration - which for loops catch silently (same protocol as Day 1 loops)

> **Trick to remember:** with = try/finally in a costume. A generator is a paused function that remembers everything.

In [ ]:
def demo():
    print("A: started")
    yield 1
    print("B: resumed")
    yield 2
    print("C: finishing")

g = demo()               # NOTHING prints - function body has not started
print("object made")
print(next(g))           # runs to first yield -> A, then 1
print(next(g))           # resumes exactly after yield 1 -> B, then 2
try:
    next(g)              # C prints, then StopIteration
except StopIteration:
    print("exhausted")


## 7. Time and Space Complexity

- try/except: near-zero cost when NO exception occurs; raising/catching costs more - exceptions are for exceptional cases, not routine control flow
- with: O(1) overhead (one enter + one exit call)
- Generators: O(1) memory regardless of stream length - THE headline; time is O(n) overall like any full pass
- List materialization vs generator: list(gen) converts O(1) memory into O(n) - only do it when you truly need all values at once

## 8. Common Mistakes

- Bare except: (catches EVERYTHING including Ctrl+C and typos in your own code) - always name the exception type
- Silently swallowing errors with except: pass - failures vanish, debugging becomes a nightmare; at minimum log them
- Opening files without with, then leaking handles when an error skips your manual close()
- Trying len(generator) or generator[0] - generators have no length and no indexing; they only know "next"
- Iterating a generator twice - second pass yields nothing (exhausted), a silent bug from Day 1 that applies doubly here
- Catching an exception too early/broadly instead of letting it propagate to code that can actually handle it

In [ ]:
# Bare except hides even your own typos - the classic self-inflicted wound
def buggy():
    try:
        resutl = 10 / 2        # typo: 'resutl'
        return result           # NameError!
    except:                     # bare except swallows the NameError
        return "division failed"  # utterly misleading message

print(buggy())   # lies to you about what went wrong

# Correct: catch ONLY what you expect
def fixed():
    try:
        return 10 / 2
    except ZeroDivisionError:
        return "division failed"

print(fixed())


## 9. Best Practices

- Catch the MOST SPECIFIC exception that you can genuinely handle; let the rest propagate
- Keep try blocks small - wrap only the line(s) that can fail, not entire functions
- Always use with for files/connections/locks - never manual open/close
- Define custom exceptions for your domain (DataValidationError beats a generic ValueError in a large pipeline)
- Prefer generators for any single-pass processing of large data; materialize with list() only when needed
- Use finally (or a context manager) for cleanup - never rely on hoping the error will not happen

## 10. Interview Questions

**Beginner**
- Q: What is the difference between except and finally?
  A: except runs only when a matching error occurs; finally runs ALWAYS - error or no error - making it the place for cleanup.
- Q: What does the with statement do when opening files?
  A: It guarantees the file is closed when the block exits - normally or via an exception - by calling the file's __exit__ method automatically.

**Intermediate**
- Q: What is the difference between an iterator and a generator?
  A: An iterator is any object implementing __iter__ and __next__; a generator is a convenient iterator created by a function containing yield - Python writes the protocol machinery for you, including state preservation between calls.
- Q: Why is a bare except considered bad practice?
  A: It catches every exception - including programming bugs (NameError, TypeError) and system signals (KeyboardInterrupt) - hiding real problems and making failures silent and undebuggable.

**Advanced**
- Q: What arguments does __exit__ receive and what does its return value control?
  A: It receives exc_type, exc_value, and traceback (all None if the block succeeded). Returning True suppresses the exception; False/None lets it propagate after cleanup runs.
- Q: What happens, step by step, when you call a generator function and then call next() on it?
  A: The call itself executes no body code - it returns a generator object. The first next() runs the body until the first yield and returns that value, freezing the frame (all locals and the instruction pointer). Each subsequent next() resumes from that exact point; when the body finishes, StopIteration is raised, which for loops absorb as the end signal.

## 11. Practice Problems

**Easy**
1. Write a function that converts a list of strings to ints, skipping non-numeric ones with try/except and reporting how many were skipped.
2. Write a generator countdown(n) yielding n down to 1, and consume it with a for loop.

**Medium**
3. Create a custom exception NegativeSalaryError and a function that validates a list of salaries, collecting valid ones and error messages separately.
4. Write a class-based context manager OpenTag(name) that prints an opening tag on enter and a closing tag on exit - verify the closing tag still prints when the block raises an error.

**Hard**
5. Build a generator pipeline: one generator yields lines from a (simulated) log file, a second generator filters lines containing "ERROR", a third extracts timestamps. Chain them, then explain in comments why memory stays O(1) no matter the file size and what happens if you try to reuse the chain a second time.

## 12. Revision Summary

- try = attempt, except = rescue (be SPECIFIC), else = clean-path only, finally = always runs
- raise throws; custom exceptions are one-line classes inheriting Exception - name them by domain
- Bare except and except-pass are the two cardinal sins of error handling
- with guarantees __exit__ cleanup (it is try/finally underneath); __exit__ gets the error info and True suppresses it
- Iterator protocol = __iter__ + __next__ + StopIteration; generators implement it for free via yield
- yield pauses and preserves the whole function state; generators are one-shot and O(1) memory
- Production trio: errors handled per-record, resources context-managed, big data streamed via generators

> **Day 2 complete.** Next: **Day 3 - NumPy Deep Dive (Data Science / EDA focus)**